# 04a · Aggregate 2025 Cooling Assistance applications to modified ZIP code areas

**Objective.** Turn the NYC Department of Social Services (DSS) workbook of Cooling Assistance approvals and
denials by ZIP code into 2025 application counts per modified ZIP code area (MODZCTA), the geography used in
`analysis.ipynb`.

The workbook was obtained by The Margin and is **not redistributed** with this repository. This notebook is
kept so the aggregation is inspectable and rerunnable by anyone who has the file. Its two outputs are
redistributed: `../inputs/dss_applications_2025_by_modzcta.csv` and
`../inputs/dss_applications_2025_unmatched_zips.csv`. Only application totals leave this notebook; approvals,
denials and denial reasons are not used anywhere in the analysis.

Set the workbook path in the next cell (or the `DSS_WORKBOOK` environment variable).

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import os

import pandas as pd

from common import INPUTS

WORKBOOK = Path(os.environ.get("DSS_WORKBOOK", ROOT.parent / "Cooling Assistance Application Approvals and Denials by Borough (2024, 2025).xlsx"))
if not WORKBOOK.exists():
    raise FileNotFoundError(f"DSS workbook not found at {WORKBOOK}. Set DSS_WORKBOOK to its path; the file is not part of this repository.")

MANUAL_ZIP_TO_MODZCTA = {"11249": "11211"}  # Source: NYC Open Data pri4-ifjk, feature 11211, label "11211, 11249"


## 1. Read the raw pivot sheets

The sheets `Approved (2025)` and `Denied (2025)` are pivot tables: one row per ZIP code, grouped under borough
subtotal rows, with a grand total. Keeping only rows whose label is a five-digit ZIP code drops the subtotal
and total rows. The workbook records borough independently of ZIP, so the same ZIP can appear under more than
one borough; those rows are summed.

In [2]:
def read_sheet(sheet: str, value_column: str, name: str) -> pd.DataFrame:
    frame = pd.read_excel(WORKBOOK, sheet_name=sheet)
    frame["zip"] = frame.iloc[:, 0].astype(str).str.strip()
    frame = frame[frame["zip"].str.fullmatch(r"\d{5}")]
    return frame.rename(columns={value_column: name}).groupby("zip", as_index=False)[name].sum()


approved = read_sheet("Approved (2025)", "Count of Zip", "approved")
denied = read_sheet("Denied (2025)", "Grand Total", "denied")
by_zip = approved.merge(denied, on="zip", how="outer")          # a ZIP may appear on only one sheet
by_zip[["approved", "denied"]] = by_zip[["approved", "denied"]].fillna(0).astype(int)
by_zip["applications_2025"] = by_zip["approved"] + by_zip["denied"]
total = int(by_zip.applications_2025.sum())
print(f"{len(by_zip)} ZIP codes, {total:,} applications in 2025")


190 ZIP codes, 26,621 applications in 2025


## 2. Map ZIP codes to modified ZIP code areas

The DOHMH ZCTA-to-MODZCTA crosswalk assigns each ZIP to one MODZCTA. One ZIP, 11249 (North Williamsburg), is
absent from the crosswalk file; the MODZCTA dataset itself labels feature 11211 as "11211, 11249", so that
assignment is added. Counts only ever aggregate upward: no ZIP total is split between areas. ZIP codes with
no MODZCTA (outside New York City, or PO-box ZIPs) are written out rather than dropped.

In [3]:
crosswalk = pd.read_csv(INPUTS / "zcta_to_modzcta.csv", dtype=str)
lookup = dict(zip(crosswalk["ZCTA"], crosswalk["MODZCTA"])) | MANUAL_ZIP_TO_MODZCTA
by_zip["modzcta"] = by_zip["zip"].map(lookup)
matched = by_zip[by_zip.modzcta.notna()]
unmatched = by_zip[by_zip.modzcta.isna()][["zip", "applications_2025"]].sort_values("applications_2025", ascending=False)
assert matched.applications_2025.sum() + unmatched.applications_2025.sum() == total   # nothing lost in the split

by_modzcta = matched.groupby("modzcta", as_index=False)["applications_2025"].sum().sort_values("modzcta")
by_modzcta.to_csv(INPUTS / "dss_applications_2025_by_modzcta.csv", index=False)
unmatched.to_csv(INPUTS / "dss_applications_2025_unmatched_zips.csv", index=False)
print(f"{int(by_modzcta.applications_2025.sum()):,} applications matched to {len(by_modzcta)} MODZCTAs; "
      f"{int(unmatched.applications_2025.sum()):,} applications in {len(unmatched)} unmatched ZIP codes")
unmatched


26,606 applications matched to 173 MODZCTAs; 15 applications in 13 unmatched ZIP codes


,zip,applications_2025
82,10705,2
88,11096,2
78,10545,1
80,10701,1
79,10550,1
81,10703,1
83,10927,1
97,11202,1
135,11306,1
182,11453,1
